# DCT Laboratory — Volume I, Chapter 11
## Enterprise Performance Architecture
**Seed `26111`** · Companion to Chapter 11 and **AXIOM Lab 1.11** (module **AXIOM-11**) · Mirrored in `DCT_V1_Ch11_Lab.xlsx`

This laboratory reproduces **Worked Example 11.1, *Who Really Chose Digital?***, with the book's declarations, and
works the five steps of AXIOM Lab 1.11, *Auditing Meridian's Scorecard*: build the dashboard, sweep the weights,
change the yardstick, roll up a ratio, walk the frontier and move the window. The 200,000 weightings that measure
how much of the space of weights each option wins are drawn, as in the book, with seed 26111, the lab's seed
(Online Appendix 11.B.6). Every Meridian number below is the book's (Worked Example 11.1, Tables 11.4–11.6,
Figures 11.1–11.3, Section 11.8, Online Appendix 11.B); the final cell checks them. Money is in \$ million.

## The declarations (Worked Example 11.1, the situation; Worked Example 1.1; Online Appendix 1.B.4)

In Worked Example 1.1 the chief financial officer's scorecard,
$\text{Score} = \tfrac14\,(\text{Liquidity} + \text{Technology} + \text{Efficiency} - \text{Strategic risk})$,
ranked the digital option first at Year 5. The paths come from the Chapter 1 model (Online Appendix 1.B.4): four of
Meridian's eight readings move, liquidity (indexed to today $= 100$, that is \$620 million), the technology platform,
operational efficiency and strategic risk (lower is better). Each reading relaxes toward an option-specific target
at its own speed, with a trough for temporary strain and a jump when the restructuring's sale closes at Year 1,
$$x_i(t) = \bar x_i + \bigl(x_i(0) - \bar x_i\bigr)e^{-k_i t} + \sigma_i D_i\,\frac{t}{\tau_i}\,e^{1 - t/\tau_i} + J_i\,\mathbf{1}\{t \ge 1\},$$
from the state declared in Chapter 3: liquidity 100, technology 41, efficiency 71, strategic risk 64, a score of
37.0. The head of Digital Solutions presents the ranking as the case for digital; the chair asks whether the data
chose digital, or the design of the scorecard.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26111      # the lab's seed; Online Appendix 11.B.6 draws its 200,000 weightings with it

# The Chapter 1 model (Online Appendix 1.B.4); t in years
READINGS = ["liquidity", "technology", "efficiency", "strategic risk"]
TODAY = np.array([100.0, 41.0, 71.0, 64.0])     # liquidity indexed to today = 100 ($620 million); indices 0-100
SIGN = np.array([1.0, 1.0, 1.0, -1.0])          # strategic risk enters with a minus sign: lower is better
OPTIONS = ["digital", "restructuring", "turnaround"]
COLORS = ["#1F4E79", "#E65100", "#B71C1C"]      # digital, restructuring, turnaround
SALE = 1.0                                      # the restructuring's sale closes at Year 1
PARAMS = {  # each reading: target, speed a year, trough sigma*D, trough time tau (years), jump at the sale
    "digital":       [(112, 0.35, -28, 2.0, 0), (78, 0.55, 0, None, 0), (77, 0.50, -9, 1.5, 0), (51, 0.40, 14, 1.2, 0)],
    "restructuring": [(104, 0.30, 0, None, 18), (38, 0.25, 0, None, 0), (74, 0.45, 0, None, 0), (56, 0.35, 0, None, -8)],
    "turnaround":    [(106, 0.45, 0, None, 0), (33, 0.20, 0, None, 0), (79, 0.60, 0, None, 0), (67, 0.30, 0, None, 0)]}

def readings(option, t):
    # the four readings of an option at times t (years), one row per time
    t = np.atleast_1d(np.asarray(t, float))
    cols = []
    for x0, (target, k, depth, tau, jump) in zip(TODAY, PARAMS[option]):
        x = target + (x0 - target) * np.exp(-k * t) + jump * (t >= SALE)
        if depth:
            x = x + depth * (t / tau) * np.exp(1 - t / tau)
        cols.append(x)
    return np.column_stack(cols)

EQUAL = np.full(4, 0.25)
def score(x, w=EQUAL):
    # the CFO's scorecard: w . (L, T, E, -R); equal weights give (L + T + E - R)/4
    return (np.asarray(x, float) * SIGN) @ w

Y5 = np.vstack([readings(o, 5.0) for o in OPTIONS])                 # Year-5 readings, one row per option
MONTHS = np.arange(61)                                               # five years, month by month
S = np.vstack([score(readings(o, MONTHS / 12)) for o in OPTIONS])    # the scorecard of each option, monthly
print(f"today: liquidity {TODAY[0]:.0f} ($620 million), technology {TODAY[1]:.0f}, efficiency {TODAY[2]:.0f}, "
      f"strategic risk {TODAY[3]:.0f} -> score {score(TODAY):.1f}")
print(f"Chapter 1 model: {len(OPTIONS)} options x {len(READINGS)} moving readings, month by month for five years; "
      f"the sale closes at Year {SALE:.0f}")

## Panel 1 — Build the dashboard (Tables 11.2 and 11.4; Section 11.8: \$23 million a year, below 55 within three quarters)

AXIOM Lab 1.11, step 1. The performance vector has ten dimensions (Equation 11.2, Table 11.2), each a flow read off
the state and the decisions (Definition 11.1). Meridian's dashboard declares six of them with their supports on
Chapter 9's architecture (Table 11.4): financial and resilience performance at treasury, operational at Industrial
Systems and Materials, strategic at the three divisions, technological at Digital, innovation at Digital and
engineering. Each dimension needs an indicator with its error and a band. Meridian's own readings declare three
performance indicators (Table 5.2 tags efficiency, ROIC and market share as performance) and the liquidity buffer
that resilience reads (Definition 11.12); technological and innovation performance have no declared indicator yet,
and a dimension without one is blind by construction (Section 11.4). Of the CFO's four readings, only efficiency is
a performance reading: Table 5.2 tags liquidity and the technology platform as capital and strategic risk as risk.

The supports also fix Table 11.4's **lag floors**: the distance between two supports along Chapter 9's fourteen
arrows is the fewest quarters in which one dimension can move another through the state (Theorem 11.3(i)). Nothing
done in Digital can reach Industrial Systems' operations in less than **two quarters**, and no path leads from Digital
to Materials at all. Section 11.8 adds the sustainability line: skipping Meridian's capability rent, **about \$23 million a year**
(Worked Example 10.1), would add that much to reported EBITDA while capability, decaying at 2.5 percent a quarter,
falls below the floor of 55 **within three quarters**: performance delivered, and borrowed from a stock.

In [ ]:
# Table 11.2: the ten dimensions of the performance vector (Equation 11.2)
DIMENSIONS = [("P_F", "financial", "earnings, cash, returns on capital", "margin, ROIC, free cash flow"),
              ("P_O", "operational", "throughput, quality, delivery, cost", "equipment effectiveness, defects"),
              ("P_S", "strategic", "position gained: share, options", "share change, win rate"),
              ("P_H", "human", "capability applied and grown", "output per employee, retention"),
              ("P_T", "technological", "platform uptime and speed of change", "availability, cycle time"),
              ("P_I", "innovation", "validated novelty shipped", "new-product revenue share"),
              ("P_G", "governance", "quality of decisions and controls", "audit findings, incidents"),
              ("P_E", "environmental", "footprint per unit of output", "intensity metrics"),
              ("P_R", "resilience", "shock absorption, shown or ready", "recovery half-life, buffer days"),
              ("P_ESG", "sustainability", "the above, without depleting capital", "Definition 11.11")]
print(f"Table 11.2: the {len(DIMENSIONS)} dimensions of the performance vector")
for sym, name, what, ind in DIMENSIONS:
    print(f"  {sym:6s} {name:15s} {what:38s} {ind}")

# Meridian's dashboard: Table 11.4's six dimensions with their supports (Chapter 9's components), the indicator
# Meridian declares for each (Table 5.2: reading and error) and today's band, the reading plus or minus its error
PLACE = {"TRS": "treasury", "IND": "Industrial Systems", "DIG": "Digital", "MAT": "Materials", "ENG": "engineering"}
SUPPORT = {"P_F": ["TRS"], "P_O": ["IND", "MAT"], "P_S": ["IND", "DIG", "MAT"], "P_T": ["DIG"],
           "P_I": ["DIG", "ENG"], "P_R": ["TRS"]}
INDICATOR = {"P_F": ("ROIC (x6)", 8.3, 0.4, "%"), "P_O": ("operational efficiency (x5)", 71.0, 3.0, ""),
             "P_S": ("market share (x8)", 19.0, 0.8, "%"),
             "P_R": ("liquidity above the $350m floor (x1)", 620.0 - 350.0, 0.01 * 620.0, "$m")}
def show(v, unit, err=False):
    if unit == "%":
        return f"{v:.1f} pp" if err else f"{v:.1f}%"
    return f"${v:.1f}m" if unit == "$m" else f"{v:.0f}"
print("\nMeridian's dashboard: Table 11.4's six dimensions, with the indicators Meridian declares (Table 5.2)")
print(f"  {'':5s} {'support':30s} {'indicator':37s} {'today':>7s} {'error':>10s}   band today")
for dim, sup in SUPPORT.items():
    where = "the three divisions" if len(sup) == 3 else ", ".join(PLACE[c] for c in sup)
    if dim in INDICATOR:
        name, v, err, unit = INDICATOR[dim]
        print(f"  {dim:5s} {where:30s} {name:37s} {show(v, unit):>7s} {'+/-' + show(err, unit, True):>10s}   "
              f"{show(v - err, unit)} to {show(v + err, unit)}")
    else:
        print(f"  {dim:5s} {where:30s} none declared: blind until an indicator reads its support")
TAGS = [("liquidity", "ECA, capital"), ("technology platform", "ECA, capital"),
        ("operational efficiency", "EPA, performance"), ("strategic risk", "ERRA, risk")]
print("the CFO's scorecard reads " + "; ".join(f"{r} ({t})" for r, t in TAGS) + ": one performance reading in four")

# Proposition 11.1 (Online Appendix 11.B): the same financial performance, different resilience
EBITDA = 0.118 * 4200.0                  # 11.8 percent of $4.2 billion
for who, liq in (("Meridian", 620.0), ("a twin", 450.0)):
    after = liq - 150.0
    print(f"  {who:9s} EBITDA ${EBITDA:.0f} million, liquidity ${liq:.0f} million -> ${after:.0f} million after a "
          f"$150 million shock: {'above' if after >= 350 else 'below'} the $350 million floor")

# Table 11.4's lag floors: distances between supports along Chapter 9's fourteen arrows (Theorem 11.3(i))
COMPONENTS = ["MKT", "GOV", "MAT", "IND", "DIG", "ENG", "TRS"]
ARROWS = [("MKT", "IND"), ("MKT", "DIG"), ("MKT", "MAT"), ("IND", "TRS"), ("DIG", "TRS"), ("MAT", "TRS"),
          ("TRS", "IND"), ("TRS", "DIG"), ("ENG", "IND"), ("IND", "ENG"), ("ENG", "DIG"), ("DIG", "ENG"),
          ("GOV", "TRS"), ("GOV", "DIG")]
IX, NEVER = {c: i for i, c in enumerate(COMPONENTS)}, 99
DIST = np.full((7, 7), NEVER); np.fill_diagonal(DIST, 0)
for u, v in ARROWS:
    DIST[IX[u], IX[v]] = 1
for k in range(7):                       # fewest arrows from every component to every other (Floyd-Warshall)
    DIST = np.minimum(DIST, DIST[:, [k]] + DIST[[k], :])
def lag(a, b):
    # the distance from dimension a's support to dimension b's
    return int(min(DIST[IX[u], IX[v]] for u in SUPPORT[a] for v in SUPPORT[b]))
TABLE114 = {("P_F", "P_O"): ("+", 1), ("P_F", "P_S"): ("+", 1), ("P_F", "P_T"): ("+", 1), ("P_F", "P_I"): ("+", 1),
            ("P_F", "P_R"): ("+", 0), ("P_O", "P_F"): ("+", 1), ("P_O", "P_S"): ("+", 0), ("P_O", "P_R"): ("+", 1),
            ("P_S", "P_F"): ("+", 1), ("P_S", "P_I"): ("+", 0), ("P_T", "P_F"): ("+", 1), ("P_T", "P_O"): ("+", 2),
            ("P_T", "P_S"): ("+", 0), ("P_T", "P_I"): ("+", 0), ("P_T", "P_R"): ("+", 1), ("P_I", "P_F"): ("+", 1),
            ("P_I", "P_S"): ("+", 0), ("P_I", "P_T"): ("+", 0), ("P_R", "P_F"): ("-", 0), ("P_R", "P_O"): ("-", 1)}
DIMS6 = list(SUPPORT)
print("\nTable 11.4 from the architecture: declared sign / lag floor in quarters, the distance between the supports")
print("  improving " + "".join(f"{b:>7s}" for b in DIMS6))
for a in DIMS6:
    cells = ["---" if a == b else (f"{TABLE114[a, b][0]}/{lag(a, b)}" if (a, b) in TABLE114 else "") for b in DIMS6]
    print(f"  {a:9s}" + "".join(f"{c:>7s}" for c in cells))
LAG_MATCH = sum(lag(a, b) == q for (a, b), (sgn, q) in TABLE114.items())
DIG_IND, DIG_MAT = int(DIST[IX["DIG"], IX["IND"]]), int(DIST[IX["DIG"], IX["MAT"]])
print(f"{LAG_MATCH} of the {len(TABLE114)} declared lag floors equal the distances between the supports")
print(f"Digital to Industrial Systems: at least {DIG_IND} quarters; Digital to Materials: "
      f"{'no path, never' if DIG_MAT >= NEVER else DIG_MAT}")

# Section 11.8: the sustainability line (the rent of Worked Example 10.1)
DELTA_Q, C_STAR, PRICE_PT = 0.025, 58.0, 4.0     # attrition a quarter; capability held; $ million a point
RENT_YEAR = 4 * DELTA_Q * C_STAR * PRICE_PT
CAP_NO_RENT = C_STAR * (1 - DELTA_Q) ** np.arange(5)
Q_BELOW = int(np.argmax(CAP_NO_RENT < 55.0))
print(f"\nSection 11.8: the capability rent is {DELTA_Q * C_STAR:.2f} points a quarter, ${RENT_YEAR:.1f} million a year;"
      f" skipping it adds that much to reported EBITDA")
print(f"  capability with no rent, quarters 0-4: " + ", ".join(f"{c:.2f}" for c in CAP_NO_RENT)
      + f" -> below the floor of 55 in quarter {Q_BELOW}")

## Panel 2 — Check for dominance (Worked Example 11.1, Step 1; Table 11.6: 47.1, 46.1 and 38.4)

Worked Example 11.1, Step 1. At Year 5 the scorecard gives digital **47.1**, restructuring **46.1** and the turnaround
**38.4** (Online Appendix 11.B.6 lists the readings). No option dominates another: digital leads restructuring by
**36.8 points** on technology; restructuring leads digital by **26.8** on liquidity, **5.8** on strategic risk and
**0.1** on efficiency; the turnaround has the best efficiency of the three. By Theorem 11.2(ii), positive weights
exist that put either leader first, so the ranking cannot be read as a finding about the options until its weights
are read as a choice.

In [ ]:
print(f"{'Year 5':14s}" + "".join(f"{r:>16s}" for r in READINGS) + f"{'score':>9s}")
print(f"{'today':14s}" + "".join(f"{v:16.2f}" for v in TODAY) + f"{score(TODAY):9.2f}")
for o, x in zip(OPTIONS, Y5):
    print(f"{o:14s}" + "".join(f"{v:16.2f}" for v in x) + f"{score(x):9.2f}")

def dominates(a, b):
    # at least as good on every reading and better on at least one (strategic risk reversed)
    a, b = np.asarray(a) * SIGN, np.asarray(b) * SIGN
    return bool(np.all(a >= b) and np.any(a > b))
DOM = [(OPTIONS[i], OPTIONS[j]) for i in range(3) for j in range(3) if i != j and dominates(Y5[i], Y5[j])]
print("\npairs in which one option dominates another:", DOM if DOM else "none")
D = (Y5[0] - Y5[1]) * SIGN           # digital less restructuring, strategic risk with its minus sign
print("digital less restructuring, reading by reading (risk reversed):", ", ".join(f"{r} {v:+.2f}" for r, v in zip(READINGS, D)))
print(f"digital leads on technology by {D[1]:.1f}; restructuring leads on liquidity by {-D[0]:.1f}, "
      f"on strategic risk by {-D[3]:.1f}, on efficiency by {-D[2]:.1f}")
print(f"best efficiency: the {OPTIONS[int(np.argmax(Y5[:, 2]))]}, {Y5[:, 2].max():.2f}; efficiency differs between the "
      f"leaders by {abs(D[2]):.2f}, inside the reading's +/-3 error (Table 5.2)")
print(f"digital's lead at equal weights: {D @ EQUAL:.2f} points")

## Panel 3 — Sweep the weights (Theorem 11.2(ii); Figure 11.1; Online Appendix 11.B.2: a tie 1.6 points away)

Worked Example 11.1, Step 2, and AXIOM Lab 1.11, step 2. Write $\mathbf{d}$ for digital's readings less
restructuring's, strategic risk with its minus sign: $\mathbf{d} = (-26.81, 36.78, -0.09, -5.83)$, and
$\mathbf{w}\cdot\mathbf{d} = 1.01$ at equal weights. Moving $t$ of weight from technology to liquidity changes the lead
by $-t(d_T - d_L)$: **0.64 points** for each point of weight, so the lead of 1.0 is gone after **1.6 points** (26.6
percent on liquidity, 23.4 percent on technology), and with 2 points moved restructuring leads, **47.7 to 47.5**.
Moving weight from technology ties the options after 2.4 points to strategic risk or 2.7 to efficiency; since any
change of weights lowers the lead by at most $\tfrac12\sum_k\lvert\Delta w_k\rvert\,(\max_k d_k - \min_k d_k)$, no
smaller change can produce a tie (Online Appendix 11.B.2). Drawing 200,000 weightings uniformly from the simplex of
nonnegative weights summing to one, with seed 26111 as in the book, digital ranks first under **49.4 percent**,
restructuring under **49.3 percent** and the turnaround under **1.3 percent**, and only with at least 68 percent on
efficiency. That count measures how much of the space of weights each option wins; it is not a probability that
either option is better.

In [ ]:
LEAD = float(D @ EQUAL)
PER_POINT = (D[1] - D[0]) / 100                    # lead lost per point of weight moved from technology to liquidity
T_TIE = LEAD / (D[1] - D[0])                       # weight moved at the tie
def moved(t, frm=1, to=0):
    # equal weights with t moved from one reading to another (default: technology to liquidity)
    w = EQUAL.copy(); w[frm] -= t; w[to] += t
    return w
print(f"lead {LEAD:.4f}; each point of weight moved from technology to liquidity cuts it by {PER_POINT:.4f}")
print(f"tie after {100 * T_TIE:.2f} points: {100 * (0.25 + T_TIE):.1f}% on liquidity, {100 * (0.25 - T_TIE):.1f}% on technology")
print(f"{'points moved':>13s}" + "".join(f"{o:>15s}" for o in OPTIONS) + "   first")
for pts in (0.0, 1.0, 100 * T_TIE, 2.0, 5.0):
    sc = score(Y5, moved(pts / 100))
    print(f"{pts:13.2f}" + "".join(f"{v:15.2f}" for v in sc) + f"   {OPTIONS[int(np.argmax(sc))] if pts != 100 * T_TIE else 'a tie'}")
MOVES = {READINGS[j]: LEAD / (D[1] - D[j]) for j in (0, 2, 3)}
print("ties when weight moves from technology to: " + ", ".join(f"{r} {100 * t:.2f} points" for r, t in MOVES.items()))
print(f"any weight change lowers the lead by at most (1/2) sum|dw| x (max d - min d) = (1/2) sum|dw| x {D.max() - D.min():.2f};"
      f" moving t from technology to liquidity uses all of it, so {100 * T_TIE:.2f} points is the smallest tie")

rng = np.random.default_rng(SEED)                  # seed 26111, as in Online Appendix 11.B.6
WD = rng.dirichlet(np.ones(4), size=200_000)       # weightings uniform on the simplex
FIRST = np.argmax((Y5 * SIGN) @ WD.T, axis=0)
SHARES = np.bincount(FIRST, minlength=3) / len(FIRST)
TURN = WD[FIRST == 2]
print("\nshare of 200,000 weightings ranking each option first (seed 26111): "
      + ", ".join(f"{o} {100 * s:.1f}%" for o, s in zip(OPTIONS, SHARES)))
print(f"the turnaround's weightings: at least {100 * TURN[:, 2].min():.1f}% on efficiency, at most "
      f"{100 * TURN[:, 1].max():.1f}% on technology")

# Figure 11.1's slice: weight on technology and on liquidity; efficiency and strategic risk share the rest equally
g = np.linspace(0, 1, 601)
WT, WL = np.meshgrid(g, g)
REST = (1 - WT - WL) / 2
WIN = np.argmax(np.einsum("oj,abj->abo", Y5 * SIGN, np.stack([WL, WT, REST, REST], axis=-1)), axis=-1).astype(float)
WIN[REST < -1e-12] = np.nan
print(f"on the slice the turnaround ranks first at {int(np.nansum(WIN == 2))} of {int(np.sum(~np.isnan(WIN)))} grid points")
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.pcolormesh(g, g, np.ma.masked_invalid(WIN), cmap=ListedColormap(["#DCE6F0", "#FBE3D3", "#F3D6D6"]), vmin=-0.5,
              vmax=2.5, shading="nearest", rasterized=True)
c = (D[2] + D[3]) / 2                              # the tie on the slice: (D_T - c) w_T + (D_L - c) w_L + c = 0
wt = np.linspace(0, 1, 801); wl = -(c + (D[1] - c) * wt) / (D[0] - c)
keep = (wl >= 0) & (wl + wt <= 1)
ax.plot(wt[keep], wl[keep], color="#616161", lw=1)
ax.plot([0, 1], [1, 0], color="#999999", lw=0.6)
ax.plot(0.25, 0.25, "o", color=COLORS[0], ms=5, zorder=5)
ax.plot(0.25 - T_TIE, 0.25 + T_TIE, "o", mfc="white", mec="#616161", ms=5, zorder=5)
ax.annotate("CFO's equal weights:\ndigital 47.1, restructuring 46.1", xy=(0.25, 0.25), xytext=(0.30, 0.07),
            fontsize=8, color=COLORS[0], arrowprops=dict(arrowstyle="-", color=COLORS[0], lw=0.6))
ax.annotate(f"tie: {100 * T_TIE:.1f} points moved\nfrom technology to liquidity", xy=(0.25 - T_TIE, 0.25 + T_TIE),
            xytext=(0.02, 0.50), fontsize=8, color="#616161", arrowprops=dict(arrowstyle="-", color="#616161", lw=0.6))
ax.text(0.37, 0.80, "weights would sum beyond 1", fontsize=8, color="#999999")
ax.text(0.615, 0.66, "efficiency and strategic risk\nshare the rest of the weight\nequally on this slice", fontsize=8,
        va="top", color="#616161", clip_on=False)
ax.text(0.615, 0.36, "all weightings (seed 26111):\n" + "\n".join(f"{o} first {100 * s:.1f}%" for o, s in zip(OPTIONS, SHARES)),
        fontsize=8, va="top", clip_on=False)
ax.legend(handles=[Patch(facecolor="#DCE6F0", edgecolor=COLORS[0], label="digital first"),
                   Patch(facecolor="#FBE3D3", edgecolor=COLORS[1], label="restructuring first")],
          frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax.set(xlim=(0, 0.6), ylim=(0, 1), xlabel="weight on technology (share of the total)",
       ylabel="weight on liquidity (share of the total)", title="Who ranks first is a choice of weights (seed 26111)")
plt.tight_layout(); plt.show()

## Panel 4 — Change the yardstick (Theorem 11.2(iii); Online Appendix 11.B.3: 52.3 to 41.4, then 24.0 to 10.6)

Worked Example 11.1, Step 3, and AXIOM Lab 1.11, step 3. The scorecard adds the readings in their own index points,
and that is itself a choice. Rescale each reading so that the best option scores 100 and the worst 0, keep the equal
weights, and **restructuring wins, 52.3 to 41.4**: the rescaling divides each reading by its spread across the
options (26.8 for liquidity, 39.7 for technology, 5.0 for efficiency, 16.9 for strategic risk), so the 5.8-point
difference in strategic risk becomes 34 points, while the 36.8-point technology lead becomes 93 and the 26.8-point
liquidity deficit becomes 100. Score by percentage improvement on today instead and **digital wins, 24.0 to 10.6**,
because technology starts lowest and its 84 percent gain outweighs every other change. A normalization
$p_i \mapsto (p_i - m_i)/r_i$ turns the declared weights $w_i$ into effective weights $w_i/r_i$: as shares, **12, 8,
62 and 18 percent** on the first scale and **16, 38, 22 and 24 percent** on the second (Online Appendix 11.B.3). The
"equal" weights are equal on neither. Same data, same weights, opposite rankings.

The lab also switches the **reference group**, the set of options or peers the scale is drawn from. The last table
below rescales against two other groups, computed from the same Year-5 readings: the options with today's company,
and the two leaders alone.

In [ ]:
def best_to_worst(X, ref=None):
    # each reading rescaled so the best of the reference group scores 100 and the worst 0 (strategic risk reversed)
    V = np.atleast_2d(X) * SIGN
    R = V if ref is None else np.atleast_2d(ref) * SIGN
    lo, hi = R.min(axis=0), R.max(axis=0)
    return 100 * (V - lo) / (hi - lo)
def pct_on_today(X):
    # percentage improvement on today's reading (strategic risk reversed)
    return 100 * (np.atleast_2d(X) - TODAY) / TODAY * SIGN
def effective(r, w=EQUAL):
    # effective weights, as shares, of a normalization that divides reading i by r_i
    e = w / r
    return e / e.sum()
SPREAD = (Y5 * SIGN).max(axis=0) - (Y5 * SIGN).min(axis=0)
B2W, PCT = best_to_worst(Y5) @ EQUAL, pct_on_today(Y5) @ EQUAL
print(f"{'yardstick, equal weights':34s}" + "".join(f"{o:>15s}" for o in OPTIONS) + "   first")
for name, sc in (("index points (the CFO's)", score(Y5)), ("best option 100, worst 0", B2W), ("percentage improvement on today", PCT)):
    print(f"{name:34s}" + "".join(f"{v:15.2f}" for v in sc) + f"   {OPTIONS[int(np.argmax(sc))]}")
print("\nspreads across the options: " + ", ".join(f"{r} {v:.2f}" for r, v in zip(READINGS, SPREAD)))
print(f"rescaled differences, digital less restructuring: strategic risk {100 * D[3] / SPREAD[3]:.1f}, "
      f"technology {100 * D[1] / SPREAD[1]:+.1f}, liquidity {100 * D[0] / SPREAD[0]:.1f}, efficiency {100 * D[2] / SPREAD[2]:.1f}")
print(f"technology's gain on today under digital: {pct_on_today(Y5)[0, 1]:.1f}%")
EFF_B2W, EFF_PCT = effective(SPREAD), effective(TODAY / 100)
for name, e in (("best 100, worst 0", EFF_B2W), ("percentage on today", EFF_PCT)):
    print(f"effective weights, {name:20s}: " + ", ".join(f"{r} {100 * v:.1f}%" for r, v in zip(READINGS, e)))
print(f"on the percentage scale digital leads by {PCT[0] - PCT[1]:.1f} points")

# Theorem 11.2(iii)'s own example: weights (1, 1), A = (3, 1), B = (1, 2), rescaled by a = (1, 3)
A_, B_, a_ = np.array([3.0, 1.0]), np.array([1.0, 2.0]), np.array([1.0, 3.0])
print(f"\nTheorem 11.2(iii): A scores {A_.sum():.0f} > {B_.sum():.0f} raw; after the rescaling (1, 3), {a_ @ A_:.0f} < {a_ @ B_:.0f}")

# Two other reference groups (computed from the book's Year-5 readings; not printed in the book)
GROUPS = {"the three options (the book's)": (Y5, [0, 1, 2]),
          "the options and today's company": (np.vstack([Y5, TODAY]), [0, 1, 2]),
          "the two leaders alone": (Y5[:2], [0, 1])}
REF = {}
print(f"\n{'best 100, worst 0, against':34s}" + "".join(f"{o:>15s}" for o in OPTIONS) + "   first   effective weight on efficiency")
for name, (grp, members) in GROUPS.items():
    sc = best_to_worst(Y5[members], grp) @ EQUAL
    full = np.full(3, np.nan); full[members] = sc
    r = (np.atleast_2d(grp) * SIGN).max(axis=0) - (np.atleast_2d(grp) * SIGN).min(axis=0)
    REF[name] = (full, effective(r))
    print(f"{name:34s}" + "".join(f"{v:15.2f}" if not np.isnan(v) else f"{'-':>15s}" for v in full)
          + f"   {OPTIONS[int(np.nanargmax(full))]:13s} {100 * effective(r)[2]:.1f}%")
print(f"against the two leaders, efficiency's {abs(D[2]):.2f}-point gap, inside its +/-3 error, counts as much as the "
      f"{D[1]:.1f}-point technology lead")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
groups = [("index points\n(the CFO's)", score(Y5)), ("best 100, worst 0\n(three options)", B2W),
          ("percentage improvement\non today", PCT), ("best 100, worst 0\n(two leaders alone)", REF["the two leaders alone"][0])]
for j, (lab, sc) in enumerate(groups):
    shown = [i for i in range(3) if not np.isnan(sc[i])]
    for pos, i in enumerate(shown):
        x = j + (pos - (len(shown) - 1) / 2) * 0.26
        ax.bar(x, sc[i], width=0.24, color=COLORS[i], label=OPTIONS[i] if j == 0 else None)
        ax.text(x, sc[i] + 1.2, f"{sc[i]:.1f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(range(len(groups))); ax.set_xticklabels([lab for lab, _ in groups], fontsize=8)
ax.set(ylim=(0, 85), ylabel="equal-weight score (points of each scale)",
       title="Same data, same weights, opposite rankings (seed 26111)")
ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=3)
plt.tight_layout(); plt.show()

## Panel 5 — Roll up a ratio (Theorem 11.5(i); Table 11.5; Online Appendix 11.B.5: 81.7 to 82.0 percent)

AXIOM Lab 1.11, step 4. A group ratio is the divisional ratios weighted by their shares of the denominator,
$R = \sum_v s_v r_v$, so between two periods it moves by a **rate effect** $\sum_v s_v'(r_v' - r_v)$ plus a **mix
effect** $\sum_v (s_v' - s_v)\,r_v$ (Theorem 11.5(i)). The theorem's two divisions raise their margins from 20.0 to
22.0 percent and from 15.0 to 16.0 percent while revenue shifts toward the second, and the group margin falls from
**18.3 to 18.0 percent**: a rate effect of $+1.33$ points and a mix effect of $-1.67$. The bank of Table 11.5 runs the
same arithmetic on a cost–income ratio, where lower is better: retail shrinks on purpose (80 to 78 percent), markets
grows (85 to 84 percent), and the group ratio worsens from **81.7 to 82.0 percent**, a rate effect of $-1.33$ points
and a mix effect of $+1.67$. The unweighted average of the two divisional ratios falls from 82.5 to 81.0 percent and
would report an improvement of 1.5 points (Online Appendix 11.B.5). The safe rule: add numerators and denominators
separately, then divide, and show the mix as its own exhibit.

The book declares no divisional margins for Meridian, so the mix bench runs on the theorem's two divisions; for
Meridian the trap is not hypothetical, since the digital option shifts revenue toward a growing division whose margins
are still being built. Parts (ii) and (iii) close the audit: three targets on a shared capacity of 10 can each fit
pairwise and fail together, and a ratio rolls up only with its denominators.

In [ ]:
def bridge(N0, D0, N1, D1):
    # group ratio before and after, and Theorem 11.5(i)'s rate and mix effects
    N0, D0, N1, D1 = (np.asarray(a, float) for a in (N0, D0, N1, D1))
    r0, r1, s0, s1 = N0 / D0, N1 / D1, D0 / D0.sum(), D1 / D1.sum()
    return {"R0": N0.sum() / D0.sum(), "R1": N1.sum() / D1.sum(), "r0": r0, "r1": r1, "s0": s0, "s1": s1,
            "rate": float(s1 @ (r1 - r0)), "mix": float((s1 - s0) @ r0), "avg0": r0.mean(), "avg1": r1.mean()}
TWO = bridge([40, 15], [200, 100], [22, 32], [100, 200])       # Theorem 11.5(i): profit and revenue of two divisions
BANK = bridge([160, 85], [200, 100], [78, 168], [100, 200])    # Table 11.5: costs over income, retail and markets
for name, b, what in (("Theorem 11.5(i), two divisions", TWO, "margin"), ("Table 11.5, the bank", BANK, "cost-income ratio")):
    print(f"{name}: divisional {what}s " + ", ".join(f"{100 * x:.1f}% -> {100 * y:.1f}%" for x, y in zip(b["r0"], b["r1"]))
          + f"; shares of the denominator " + ", ".join(f"{x:.2f} -> {y:.2f}" for x, y in zip(b["s0"], b["s1"])))
    print(f"  group {100 * b['R0']:.2f}% -> {100 * b['R1']:.2f}% ({100 * (b['R1'] - b['R0']):+.2f} points) = rate effect "
          f"{100 * b['rate']:+.2f} + mix effect {100 * b['mix']:+.2f}; the unweighted average says "
          f"{100 * b['avg0']:.1f}% -> {100 * b['avg1']:.1f}%")

# The mix bench: hold this year's divisional margins and move the share of revenue
SHARE = np.linspace(0, 1, 101)
GROUP_NEW = SHARE * TWO["r1"][0] + (1 - SHARE) * TWO["r1"][1]
GROUP_OLD = SHARE * TWO["r0"][0] + (1 - SHARE) * TWO["r0"][1]
CROSS = (TWO["R0"] - TWO["r1"][1]) / (TWO["r1"][0] - TWO["r1"][1])
print(f"\nmix bench (computed from the theorem's example): at this year's margins the group beats last year's "
      f"{100 * TWO['R0']:.2f}% only while division 1 keeps at least {100 * CROSS:.1f}% of revenue; it keeps "
      f"{100 * TWO['s1'][0]:.1f}%")

# Theorem 11.5(ii): targets on a capacity shared by three units; (iii): ratios roll up only with denominators
K_CAP, TAU = 10, np.array([4, 4, 4])
PAIRS = [int(TAU[i] + TAU[j]) for i in range(3) for j in range(i + 1, 3)]
print(f"shared capacity {K_CAP}: every pair of targets fits ({max(PAIRS)} <= {K_CAP}); all three do not ({TAU.sum()} > {K_CAP})")
c_ = TWO["r0"]
print(f"same divisional ratios {100 * c_[0]:.0f}% and {100 * c_[1]:.0f}%: group {100 * c_.mean():.2f}% with equal revenue, "
      f"{100 * (2 * c_[0] + c_[1]) / 3:.2f}% when division 1's revenue doubles: no certificate without the denominators")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.axvspan(0, 100 * CROSS, color="#F3D6D6", lw=0)
ax.plot(100 * SHARE, 100 * GROUP_OLD, color="#616161", lw=1.4, ls="--", label="last year's margins: 20.0% and 15.0%")
ax.plot(100 * SHARE, 100 * GROUP_NEW, color=COLORS[0], lw=2, label="this year's margins: 22.0% and 16.0%")
ax.axhline(100 * TWO["R0"], color="#999999", lw=0.6)
ax.plot(100 * TWO["s0"][0], 100 * TWO["R0"], "o", color="#616161", ms=6, zorder=5)
ax.plot(100 * TWO["s1"][0], 100 * TWO["R1"], "o", color=COLORS[2], ms=6, zorder=5)
ax.annotate(f"last year: share {100 * TWO['s0'][0]:.0f}%, group {100 * TWO['R0']:.1f}%", xy=(100 * TWO["s0"][0], 100 * TWO["R0"]),
            xytext=(70, 17.0), fontsize=8, color="#616161", arrowprops=dict(arrowstyle="-", color="#616161", lw=0.6))
ax.annotate(f"this year: share {100 * TWO['s1'][0]:.0f}%, group {100 * TWO['R1']:.1f}%", xy=(100 * TWO["s1"][0], 100 * TWO["R1"]),
            xytext=(1.5, 20.2), fontsize=8, color=COLORS[2], arrowprops=dict(arrowstyle="-", color=COLORS[2], lw=0.6))
ax.text(100 * CROSS / 2, 22.35, f"group below last year's {100 * TWO['R0']:.1f}%\nwhile the share is under {100 * CROSS:.1f}%",
        ha="center", va="top", fontsize=8, color=COLORS[2])
ax.set(xlim=(0, 100), ylim=(15, 22.5), xlabel="share of revenue in division 1, the higher-margin division (%)",
       ylabel="group margin (%)", title="Every division improves while the group falls (seed 26111)")
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

## Panel 6 — Walk the frontier (Theorem 11.4; Proposition 11.2; Figure 11.2: 0.84 months of cover a point)

AXIOM Lab 1.11, step 5, first part. Every option that maximizes a positively weighted index is Pareto optimal
(Theorem 11.4(ii)), but the converse fails when programs are indivisible. Of the programs $(0, 10)$, $(4, 4)$ and
$(10, 0)$, in points of margin and months of resilience gained, the balanced one is Pareto optimal, yet every positive
weighting scores an end program higher, since $\max(10w_1, 10w_2) \ge 5(w_1 + w_2) > 4(w_1 + w_2)$: no index ever ranks
it first (Theorem 11.4(iii)). Three programs $(0, m)$, $(a, a)$ and $(m, 0)$ leave the balanced one visible exactly when
$2a \ge m$: at $(5, 5)$ it ties at equal weights, at $(6, 6)$ equal weights rank it first (Online Appendix 11.B.4). On
Figure 11.2's smooth frontier of margin against liquidity cover, an operating point inside the frontier can improve
both before any trade-off arises (Proposition 11.2); at the frontier point reached from it, the slope prices the
trade: **one point of margin costs 0.84 months of cover**, the exchange rate $-w_1/w_2$ of the weights that choose it
(Theorem 11.4(iv)). Meridian's three options are all Pareto optimal on the scorecard's four readings, and each ranks
first under some weights (Panel 3): none of them is invisible.

In [ ]:
def pareto(P):
    # indices of the points no other point dominates
    P = np.asarray(P, float)
    return [i for i in range(len(P)) if not any(np.all(P[j] >= P[i]) and np.any(P[j] > P[i]) for j in range(len(P)))]
def first_count(P, n=999):
    # how many of n positive weightings (w, 1 - w) rank each program first
    w1 = np.linspace(0.001, 0.999, n)
    top = np.argmax(np.asarray(P, float) @ np.vstack([w1, 1 - w1]), axis=0)
    return np.bincount(top, minlength=len(P))
PROGRAMS = np.array([[0.0, 10.0], [4.0, 4.0], [10.0, 0.0]])          # Theorem 11.4(iii): margin, resilience gain
print("Pareto optimal: " + ", ".join(f"({PROGRAMS[i][0]:.0f}, {PROGRAMS[i][1]:.0f})" for i in pareto(PROGRAMS)))
BAL_FIRST = int(first_count(PROGRAMS)[1])
print(f"of 999 positive weightings, {BAL_FIRST} rank the balanced program first; at equal weights the scores are "
      + ", ".join(f"{v:.0f}" for v in PROGRAMS @ [0.5, 0.5]))
for a in (4.0, 5.0, 6.0):                                             # Online Appendix 11.B.4: visible iff 2a >= m
    P = np.array([[0.0, 10.0], [a, a], [10.0, 0.0]])
    eq = P @ [0.5, 0.5]
    print(f"  balanced program ({a:.0f}, {a:.0f}): 2a {'>=' if 2 * a >= 10 else '<'} 10, equal weights score it {eq[1]:.0f} against "
          f"{eq[0]:.0f} -> {'first' if eq[1] > eq[0] else ('a tie' if eq[1] == eq[0] else 'never first')}")

# Figure 11.2, left: the frontier (m/16)^2 + (r/12)^2 = 1 of EBITDA margin (%) and liquidity cover (months)
MX, RY, P0 = 16.0, 12.0, np.array([9.0, 6.0])
GROW = 1 / np.sqrt((P0[0] / MX) ** 2 + (P0[1] / RY) ** 2)
PF = GROW * P0
SLOPE = -(RY ** 2 * PF[0]) / (MX ** 2 * PF[1])
print(f"\noperating point ({P0[0]:.0f}, {P0[1]:.0f}) is inside: ({P0[0] / MX:.4f})^2 + ({P0[1] / RY:.4f})^2 = "
      f"{(P0[0] / MX) ** 2 + (P0[1] / RY) ** 2:.4f} < 1, so both can improve (Proposition 11.2)")
print(f"expanding both by {GROW:.4f} reaches the frontier at ({PF[0]:.2f}% margin, {PF[1]:.2f} months of cover)")
print(f"slope there {SLOPE:.4f}: one point of margin costs {-SLOPE:.2f} months of cover; the weights that choose this "
      f"point are in the ratio w1/w2 = {-SLOPE:.4f}")
print("\nMeridian's options on the scorecard's four readings, all Pareto optimal: "
      + ", ".join(f"{OPTIONS[i]} (first under {100 * SHARES[i]:.1f}% of weightings)" for i in pareto(Y5 * SIGN)))

## Panel 7 — Move the window (Theorem 11.3(iii); Figure 11.3; Online Appendix 11.B.6: months 11, 29, 33 and 56)

Worked Example 11.1, Step 4, and AXIOM Lab 1.11, step 5, second part. Run the scorecard month by month. Digital's
score first falls, to **32.11 at month 11**, and does not regain today's 37.0 until **month 29**: the J-curve of
Theorem 11.3(iii), driven by the liquidity trough and the efficiency dip while the platform is built. It passes the
turnaround at **month 33** and restructuring only at **month 56**. Annual reviews in Years 1 and 2 rank digital last,
those in Years 3 and 4 rank it second; only a five-year window ranks it first, so a board that compares the options
at its annual reviews will see digital trail for four years. Restructuring's score jumps by 6.57 points when the sale
closes at Year 1; before that, restructuring and the turnaround differ by less than 0.1 point.

The theorem prices the window itself. A program that spends $\bar u$ in one period, at a cost $c\bar u$, and gains
$\Delta$ in every period after it shows a net gain over $H$ periods exactly when $(H - 1)\Delta > c\bar u$
(Equation 11.3), so any window shorter than $1 + c\bar u/\Delta$ periods reports a sound program as a loss. When the
built state decays, the window lengthens: a program that costs 10 and adds 3 in its first period of benefit pays back
over windows longer than **4.3** periods if the benefit lasts, **4.8** if it decays at 10 percent a period and **5.9** at
20 percent; at 30 percent or more it never pays back (Online Appendix 11.B.1).

In [ ]:
TODAY_SCORE = float(score(TODAY))
LOW = int(np.argmin(S[0]))
BELOW = MONTHS[S[0] < TODAY_SCORE]
BACK = int(BELOW[-1] + 1)
PASS_T, PASS_R = int(np.argmax(S[0] > S[2])), int(np.argmax(S[0] > S[1]))
print(f"digital: low {S[0, LOW]:.2f} at month {LOW}; below today's {TODAY_SCORE:.1f} from month {BELOW[0]} to month "
      f"{BELOW[-1]}; {S[0, BACK]:.2f} at month {BACK}")
print(f"passes the turnaround at month {PASS_T} ({S[0, PASS_T]:.2f} against {S[2, PASS_T]:.2f}) and restructuring at "
      f"month {PASS_R} ({S[0, PASS_R]:.2f} against {S[1, PASS_R]:.2f})")
JUMP = S[1, 12] - S[1, 11]
PRE_GAP = float(np.abs(S[1, :12] - S[2, :12]).max())
print(f"restructuring jumps by {JUMP:.2f} points when the sale closes; before it, restructuring and the turnaround differ "
      f"by at most {PRE_GAP:.3f}")
RANK = 1 + (S > S[0]).sum(axis=0)                  # digital's rank at each month
print(f"\n{'review':>8s}" + "".join(f"{o:>15s}" for o in OPTIONS) + "   first          digital ranks")
for yr in range(1, 6):
    m = 12 * yr
    print(f"{'Year ' + str(yr):>8s}" + "".join(f"{v:15.2f}" for v in S[:, m]) + f"   {OPTIONS[int(np.argmax(S[:, m]))]:14s} {RANK[m]}")
print("\nslide the review date (month): digital against today and against the leader")
for m in (12, 24, 28, 29, 32, 33, 55, 56, 60):
    print(f"  month {m:2d}: digital {S[0, m]:6.2f}, {'a gain' if S[0, m] > TODAY_SCORE else 'a loss'} on today's "
          f"{TODAY_SCORE:.1f}; rank {RANK[m]}; leader {OPTIONS[int(np.argmax(S[:, m]))]} {S[:, m].max():.2f}")

# Theorem 11.3(iii) and Online Appendix 11.B.1: the window a one-off program needs
COST, GAIN = 10.0, 3.0                             # the program costs 10 and adds 3 in its first period of benefit
def payback(delta):
    # the window beyond which the program shows a net gain (Equation 11.3 when delta = 0); None if it never does
    if delta == 0:
        return 1 + COST / GAIN
    x = COST * delta / GAIN
    return None if x >= 1 - 1e-12 else 1 + np.log(1 / (1 - x)) / np.log(1 / (1 - delta))
def cumulative(H, delta):
    # the program's cumulative effect over periods 1 to H: the cost now, then a gain that decays at delta
    return -COST + sum(GAIN * (1 - delta) ** (k - 2) for k in range(2, H + 1))
WINDOWS = {delta: payback(delta) for delta in (0.0, 0.1, 0.2, 0.3)}
print(f"\nOnline Appendix 11.B.1, cost 10 and first gain 3: {'decay':>6s} {'window':>8s}   cumulative effect over H = 1 ... 8")
for delta, h in WINDOWS.items():
    print(f"{'':44s}{delta:6.0%} {('never' if h is None else f'{h:.2f}'):>8s}   "
          + " ".join(f"{cumulative(H, delta):6.2f}" for H in range(1, 9)))

fig, ax = plt.subplots(figsize=(7.6, 3.6))
for m in (12, 24, 36, 48, 60):
    ax.axvline(m, color="#E0E0E0", lw=0.8, zorder=0)
ax.axhline(TODAY_SCORE, color="#999999", lw=0.8, ls="--")
ax.fill_between(MONTHS, S[0], TODAY_SCORE, where=S[0] < TODAY_SCORE, color="#DCE6F0", lw=0)
ax.plot(MONTHS, S[0], color=COLORS[0], lw=2, label="digital")
ax.plot(MONTHS[:12], S[1, :12], color=COLORS[1], lw=2, label="restructuring")
ax.plot(MONTHS[11:13], S[1, 11:13], color=COLORS[1], lw=1, ls=":")
ax.plot(MONTHS[12:], S[1, 12:], color=COLORS[1], lw=2)
ax.plot(MONTHS, S[2], color=COLORS[2], lw=2, label="turnaround")
ax.plot(LOW, S[0, LOW], "o", color=COLORS[0], ms=5)
ax.annotate(f"low {S[0, LOW]:.1f} at month {LOW}; back to\ntoday's {TODAY_SCORE:.1f} at month {BACK}", xy=(LOW, S[0, LOW]),
            xytext=(16, 31.0), fontsize=8, color=COLORS[0], arrowprops=dict(arrowstyle="-", color=COLORS[0], lw=0.6))
for m, other in ((PASS_T, 2), (PASS_R, 1)):
    ax.plot(m, S[0, m], "o", mfc="white", mec=COLORS[0], ms=5, zorder=5)
ax.annotate(f"passes the turnaround\nat month {PASS_T}", xy=(PASS_T, S[0, PASS_T]), xytext=(34, 34.2), fontsize=8,
            color=COLORS[0], arrowprops=dict(arrowstyle="-", color=COLORS[0], lw=0.6))
ax.annotate(f"passes restructuring\nat month {PASS_R}", xy=(PASS_R, S[0, PASS_R]), xytext=(38, 49.3), fontsize=8,
            color=COLORS[0], arrowprops=dict(arrowstyle="-", color=COLORS[0], lw=0.6))
ax.annotate(f"sale closes: +{JUMP:.1f}", xy=(12, 41.5), xytext=(1.5, 44.5), fontsize=8, color=COLORS[1],
            arrowprops=dict(arrowstyle="-", color=COLORS[1], lw=0.6))
ax.text(60, TODAY_SCORE - 0.4, f"today's score {TODAY_SCORE:.1f}", ha="right", va="top", fontsize=8, color="#616161")
ax.set_xticks([0, 12, 24, 36, 48, 60]); ax.set_yticks([30, 35, 40, 45, 50])
ax.set(xlim=(0, 60), ylim=(30, 52), xlabel="months from today (annual reviews at 12, 24, 36, 48 and 60)",
       ylabel="CFO's scorecard (index points)", title="Digital trails restructuring until month 56 (seed 26111)")
ax.legend(frameon=False, fontsize=8, loc="lower right", ncol=3)
plt.tight_layout(); plt.show()

## Panel 8 — Declare the apparatus (Worked Example 11.1, Step 5; Tables 11.3 and 11.6; Worked Example 8.1: 24.6 and 3.0 percent)

Worked Example 11.1, Step 5. Table 11.6 collects the readings: the same three options, six declarations, and the
first place moving between digital and restructuring. The strategy team's recommendation is therefore not "digital
scores 47.1" but a declaration the board can own: the weights, with their 1.6-point margin from a tie; the yardstick and
the options or peers it rescales against; the aggregation rule and any gates; and the review window, at least five
years for the digital program. Each is a legitimate choice. None is a fact about the options.

The worked example's discussion asks whether the board's risk appetite, at most a 5 percent chance of breaching the
liquidity floor, should enter the scorecard as a weight or as a gate that an option must pass before it is scored
(Table 11.3). The odds are Worked Example 8.1's, drawn as in the book with **random seed 8** (Volume II, Online Appendix
1.B.4) from Chapter 8's uncertainty model: the digital program's quarterly liquidity plan over three years, a deviation
that carries 60 percent of each quarter's surprise (standard deviation \$50 million) into the next, and funding-window
closures about once a decade that remove on average \$120 million. The program as planned breaches the \$350 million
floor in **24.6 percent** of 100,000 paths; a standby line that replaces up to \$100 million of any closure, together with
a phased build, brings the chance to **3.0 percent**. A weight would let technology buy back the breach risk at a fixed
rate; a gate, the lexicographic rule of Table 11.3, fails the program as planned in its tier and admits it once the
levers bring it within the appetite.

In [ ]:
TABLE116 = [("The CFO's scorecard", score(Y5)),
            ("2 points of weight moved from technology to liquidity", score(Y5, moved(0.02))),
            ("Best option 100, worst 0", B2W),
            ("Percentage improvement on today", PCT),
            ("Review at Year 1", S[:, 12]),
            ("Review at Year 3", S[:, 36])]
print(f"Table 11.6 {'declaration':55s}" + "".join(f"{o:>15s}" for o in OPTIONS) + "   first")
for label, sc in TABLE116:
    print(f"  {label:63s}" + "".join(f"{v:15.1f}" for v in sc) + f"   {OPTIONS[int(np.argmax(sc))]}")
print("first place, row by row: " + ", ".join(OPTIONS[int(np.argmax(sc))] for _, sc in TABLE116))

# Worked Example 8.1's uncertainty model, drawn as in the book with random seed 8 (Volume II, Online Appendix 1.B.4)
BOOK_SEED, N_PATHS, QUARTERS = 8, 100_000, 12
FLOOR, APPETITE = 350.0, 0.05                    # $ million; the board's appetite for a breach
A_E, SD_W = 0.6, 50.0                            # deviation E' = 0.6 E + w, surprises w with sd $50 million
LAM, J_MEAN, J_SD = 0.10, 120.0, 50.0            # closures a year; their size, mean and sd ($ million, gamma)
LINE, DEPTH, DEPTH_PHASED = 100.0, 28.0, 20.0    # standby line per closure; the plan's draw depth, planned and phased

def plan(depth=DEPTH):
    # the digital program's liquidity, quarterly for three years: Chapter 1's index at $6.2 million a point
    t = np.arange(QUARTERS + 1) / 4
    return 6.2 * (112 + (100 - 112) * np.exp(-0.35 * t) - depth * (t / 2) * np.exp(1 - t / 2))

def breach_odds(line=0.0, depth=DEPTH, n=N_PATHS, seed=BOOK_SEED):
    # share of paths below the floor in some quarter; each quarter the n surprises, the n closure counts, then the
    # sizes of that quarter's closures in path order, as in the book's engine
    rng = np.random.default_rng(seed)
    pl, E, J, low = plan(depth), np.zeros(n), np.zeros(n), np.full(n, np.inf)
    shape, scale = (J_MEAN / J_SD) ** 2, J_SD ** 2 / J_MEAN
    for k in range(QUARTERS):
        w = SD_W * rng.standard_normal(n)
        cnt = rng.poisson(LAM / 4, n)
        sizes = rng.gamma(shape, scale, cnt.sum())
        J = J - np.bincount(np.repeat(np.arange(n), cnt), np.maximum(sizes - line, 0.0), minlength=n)
        E = A_E * E + w
        low = np.minimum(low, pl[k + 1] + E + J)
    return float((low < FLOOR).mean())

P_PLAN, P_LEVERS = breach_odds(), breach_odds(line=LINE, depth=DEPTH_PHASED)
print(f"\nliquidity plan: ${plan()[0]:.0f} million today, trough ${plan().min():.0f} million in quarter {plan().argmin()}; "
      f"phased build trough ${plan(DEPTH_PHASED).min():.0f} million")
print(f"{'gate: P(breach) <= 5% (seed 8, 100,000 paths)':50s} {'P(breach)':>10s}   verdict")
for name, p in (("digital program as planned", P_PLAN), ("with the standby line and the phased build", P_LEVERS)):
    print(f"  {name:48s} {100 * p:9.1f}%   {'passes: scored' if p <= APPETITE else 'fails its tier: not scored'}")

## Validation — the book's numbers, and agreement with `DCT_V1_Ch11_Lab.xlsx`

Each checkpoint is a number printed in Chapter 11 (Worked Example 11.1, Tables 11.4–11.6, Figures 11.1–11.3, the
theorems' examples, Section 11.8) or in Online Appendix 11.B, with the readings of Online Appendix 11.B.6 and the odds of
Worked Example 8.1 that its discussion cites. The workbook recomputes the ones marked *workbook* with Excel formulas and
shows PASS against the same engine values on its `Checks` sheet.

In [ ]:
YR = {yr: S[:, 12 * yr] for yr in range(1, 6)}
checks = [
    ("today's score (workbook)", TODAY_SCORE, 37.0, 0.05),
    ("Year 5: digital liquidity (workbook)", Y5[0, 0], 94.30, 0.005),
    ("Year 5: digital technology, Table 1.9 (workbook)", Y5[0, 1], 75.6, 0.05),
    ("Year 5: digital efficiency (workbook)", Y5[0, 2], 73.60, 0.005),
    ("Year 5: digital strategic risk (workbook)", Y5[0, 3], 55.22, 0.005),
    ("Year 5: restructuring liquidity (workbook)", Y5[1, 0], 121.11, 0.005),
    ("Year 5: restructuring technology (workbook)", Y5[1, 1], 38.86, 0.005),
    ("Year 5: restructuring efficiency (workbook)", Y5[1, 2], 73.68, 0.005),
    ("Year 5: restructuring strategic risk (workbook)", Y5[1, 3], 49.39, 0.005),
    ("Year 5: turnaround liquidity (workbook)", Y5[2, 0], 105.37, 0.005),
    ("Year 5: turnaround technology (workbook)", Y5[2, 1], 35.94, 0.005),
    ("Year 5: turnaround efficiency (workbook)", Y5[2, 2], 78.60, 0.005),
    ("Year 5: turnaround strategic risk (workbook)", Y5[2, 3], 66.33, 0.005),
    ("Year 5 score: digital (workbook)", score(Y5[0]), 47.08, 0.005),
    ("Year 5 score: restructuring (workbook)", score(Y5[1]), 46.07, 0.005),
    ("Year 5 score: turnaround (workbook)", score(Y5[2]), 38.40, 0.005),
    ("pairs with one option dominating (workbook)", len(DOM), 0, 0),
    ("digital's technology lead (workbook)", D[1], 36.8, 0.05),
    ("restructuring's liquidity lead (workbook)", -D[0], 26.8, 0.05),
    ("restructuring's strategic-risk lead (workbook)", -D[3], 5.8, 0.05),
    ("restructuring's efficiency lead (workbook)", -D[2], 0.1, 0.05),
    ("digital's lead at equal weights (workbook)", LEAD, 1.01, 0.005),
    ("lead lost per point of weight moved (workbook)", PER_POINT, 0.64, 0.005),
    ("points of weight moved to tie (workbook)", 100 * T_TIE, 1.6, 0.05),
    ("tie: weight on liquidity, % (workbook)", 100 * (0.25 + T_TIE), 26.6, 0.05),
    ("tie: weight on technology, % (workbook)", 100 * (0.25 - T_TIE), 23.4, 0.05),
    ("2 points moved: digital (workbook)", score(Y5[0], moved(0.02)), 47.5, 0.05),
    ("2 points moved: restructuring (workbook)", score(Y5[1], moved(0.02)), 47.7, 0.05),
    ("2 points moved: turnaround (workbook)", score(Y5[2], moved(0.02)), 39.8, 0.05),
    ("tie moving weight to strategic risk, points (workbook)", 100 * MOVES["strategic risk"], 2.4, 0.05),
    ("tie moving weight to efficiency, points (workbook)", 100 * MOVES["efficiency"], 2.7, 0.05),
    ("weightings ranking digital first, % (seed 26111)", 100 * SHARES[0], 49.4, 0.05),
    ("weightings ranking restructuring first, % (seed 26111)", 100 * SHARES[1], 49.3, 0.05),
    ("weightings ranking the turnaround first, % (seed 26111)", 100 * SHARES[2], 1.3, 0.05),
    ("turnaround first: least weight on efficiency, %", 100 * TURN[:, 2].min(), 68, 0.5),
    ("turnaround first: most weight on technology, %", 100 * TURN[:, 1].max(), 13, 0.5),
    ("best 100, worst 0: digital (workbook)", B2W[0], 41.4, 0.05),
    ("best 100, worst 0: restructuring (workbook)", B2W[1], 52.3, 0.05),
    ("best 100, worst 0: turnaround (workbook)", B2W[2], 35.3, 0.05),
    ("spread of liquidity (workbook)", SPREAD[0], 26.8, 0.05),
    ("spread of technology (workbook)", SPREAD[1], 39.7, 0.05),
    ("spread of efficiency (workbook)", SPREAD[2], 5.0, 0.05),
    ("spread of strategic risk (workbook)", SPREAD[3], 16.9, 0.05),
    ("rescaled strategic-risk difference (workbook)", -100 * D[3] / SPREAD[3], 34, 0.5),
    ("rescaled technology lead (workbook)", 100 * D[1] / SPREAD[1], 93, 0.5),
    ("rescaled liquidity deficit (workbook)", -100 * D[0] / SPREAD[0], 100, 0.5),
    ("percentage on today: digital (workbook)", PCT[0], 24.0, 0.05),
    ("percentage on today: restructuring (workbook)", PCT[1], 10.6, 0.05),
    ("percentage on today: turnaround (workbook)", PCT[2], 0.0, 0.05),
    ("technology's gain on today under digital, % (workbook)", pct_on_today(Y5)[0, 1], 84, 0.5),
    ("percentage scale: digital's lead (workbook)", PCT[0] - PCT[1], 13, 0.5),
    ("effective weight, best-worst: liquidity, % (workbook)", 100 * EFF_B2W[0], 12, 0.5),
    ("effective weight, best-worst: technology, % (workbook)", 100 * EFF_B2W[1], 8, 0.5),
    ("effective weight, best-worst: efficiency, % (workbook)", 100 * EFF_B2W[2], 62, 0.5),
    ("effective weight, best-worst: risk, % (workbook)", 100 * EFF_B2W[3], 18, 0.5),
    ("effective weight, percentage: liquidity, % (workbook)", 100 * EFF_PCT[0], 16, 0.5),
    ("effective weight, percentage: technology, % (workbook)", 100 * EFF_PCT[1], 38, 0.5),
    ("effective weight, percentage: efficiency, % (workbook)", 100 * EFF_PCT[2], 22, 0.5),
    ("effective weight, percentage: risk, % (workbook)", 100 * EFF_PCT[3], 24, 0.5),
    ("Theorem 11.2(iii): A after rescaling (workbook)", a_ @ A_, 6, 0),
    ("Theorem 11.2(iii): B after rescaling (workbook)", a_ @ B_, 7, 0),
    ("two divisions: group margin before, % (workbook)", 100 * TWO["R0"], 18.33, 0.005),
    ("two divisions: group margin after, % (workbook)", 100 * TWO["R1"], 18.0, 0.05),
    ("two divisions: rate effect, points (workbook)", 100 * TWO["rate"], 1.33, 0.005),
    ("two divisions: mix effect, points (workbook)", 100 * TWO["mix"], -1.67, 0.005),
    ("bank: group cost-income before, % (workbook)", 100 * BANK["R0"], 81.7, 0.05),
    ("bank: group cost-income after, % (workbook)", 100 * BANK["R1"], 82.0, 0.05),
    ("bank: rate effect, points (workbook)", 100 * BANK["rate"], -1.33, 0.005),
    ("bank: mix effect, points (workbook)", 100 * BANK["mix"], 1.67, 0.005),
    ("bank: unweighted average before, % (workbook)", 100 * BANK["avg0"], 82.5, 0.05),
    ("bank: unweighted average after, % (workbook)", 100 * BANK["avg1"], 81.0, 0.05),
    ("shared capacity: three targets together (workbook)", TAU.sum(), 12, 0),
    ("indivisible: weightings ranking (4, 4) first (workbook)", BAL_FIRST, 0, 0),
    ("frontier: months of cover a margin point (workbook)", -SLOPE, 0.84, 0.005),
    ("digital's low (workbook)", S[0, LOW], 32.11, 0.005),
    ("month of digital's low (workbook)", LOW, 11, 0),
    ("month digital regains today's score (workbook)", BACK, 29, 0),
    ("digital at month 29 (workbook)", S[0, BACK], 37.19, 0.005),
    ("month digital passes the turnaround (workbook)", PASS_T, 33, 0),
    ("digital at month 33 (workbook)", S[0, PASS_T], 38.74, 0.005),
    ("turnaround at month 33 (workbook)", S[2, PASS_T], 38.41, 0.005),
    ("month digital passes restructuring (workbook)", PASS_R, 56, 0),
    ("digital at month 56 (workbook)", S[0, PASS_R], 46.15, 0.005),
    ("Year 1 review: digital (workbook)", YR[1][0], 32.15, 0.005),
    ("Year 1 review: restructuring (workbook)", YR[1][1], 44.46, 0.005),
    ("Year 1 review: turnaround (workbook)", YR[1][2], 37.89, 0.005),
    ("Year 2 review: digital (workbook)", YR[2][0], 35.27, 0.005),
    ("Year 2 review: restructuring (workbook)", YR[2][1], 45.11, 0.005),
    ("Year 2 review: turnaround (workbook)", YR[2][2], 38.29, 0.005),
    ("Year 3 review: digital (workbook)", YR[3][0], 39.88, 0.005),
    ("Year 3 review: restructuring (workbook)", YR[3][1], 45.55, 0.005),
    ("Year 3 review: turnaround (workbook)", YR[3][2], 38.43, 0.005),
    ("Year 4 review: digital (workbook)", YR[4][0], 43.96, 0.005),
    ("Year 4 review: restructuring (workbook)", YR[4][1], 45.86, 0.005),
    ("Year 4 review: turnaround (workbook)", YR[4][2], 38.45, 0.005),
    ("digital's rank, Year 1 review (workbook)", RANK[12], 3, 0),
    ("digital's rank, Year 2 review (workbook)", RANK[24], 3, 0),
    ("digital's rank, Year 3 review (workbook)", RANK[36], 2, 0),
    ("digital's rank, Year 4 review (workbook)", RANK[48], 2, 0),
    ("digital's rank, Year 5 review (workbook)", RANK[60], 1, 0),
    ("restructuring's jump at the sale (workbook)", JUMP, 6.57, 0.005),
    ("payback window, no decay (workbook)", WINDOWS[0.0], 4.3, 0.05),
    ("payback window, 10% decay (workbook)", WINDOWS[0.1], 4.8, 0.05),
    ("payback window, 20% decay (workbook)", WINDOWS[0.2], 5.9, 0.05),
    ("30% decay: windows that pay back (workbook)", 0 if WINDOWS[0.3] is None else 1, 0, 0),
    ("Table 11.4: lag floors reproduced (workbook)", LAG_MATCH, 20, 0),
    ("lag floor, Digital to Industrial Systems (workbook)", DIG_IND, 2, 0),
    ("Section 11.8: capability rent, $ million a year (workbook)", RENT_YEAR, 23, 0.5),
    ("Section 11.8: quarter capability falls below 55 (workbook)", Q_BELOW, 3, 0),
    ("Proposition 11.1: EBITDA, $ million (workbook)", EBITDA, 496, 0.5),
    ("breach, digital as planned, % (seed 8)", 100 * P_PLAN, 24.6, 0.05),
    ("breach, standby line and phased build, % (seed 8)", 100 * P_LEVERS, 3.0, 0.05),
]
ok, width = 0, max(len(name) for name, *_ in checks)
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol + 1e-9 else "FAIL"; ok += status == "PASS"
    print(f"{name:{width}s} {got:10.4f}   book {want:9.4f}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 12 puts this chapter's targets, floors and frontiers together with Chapter 8's distributions and calls
the result risk: the exposure of declared objectives to the distribution of the company's state, with vulnerability,
robustness and resilience kept apart. A target without a distribution is not yet a statement about risk. AXIOM-11 runs
the same model interactively (Lab 1.11); this notebook and the workbook are its reproducible record.